In [8]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    "postgresql+psycopg://aikalee@localhost:5432/healthcare"
)

raw_df = pd.read_csv("13100961-eng/13100961.csv")

raw_df.to_sql(
    "wait_times_raw",
    engine,
    if_exists="replace",
    index=False
)

-1

In [220]:
clean_df = raw_df.copy()

In [221]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6468 entries, 0 to 6467
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   REF_DATE         6468 non-null   int64  
 1   GEO              6468 non-null   str    
 2   DGUID            5880 non-null   str    
 3   Gender           6468 non-null   str    
 4   Age group        6468 non-null   str    
 5   Indicators       6468 non-null   str    
 6   Characteristics  6468 non-null   str    
 7   UOM              6468 non-null   str    
 8   UOM_ID           6468 non-null   int64  
 9   SCALAR_FACTOR    6468 non-null   str    
 10  SCALAR_ID        6468 non-null   int64  
 11  VECTOR           6468 non-null   str    
 12  COORDINATE       6468 non-null   str    
 13  VALUE            5918 non-null   float64
 14  STATUS           1176 non-null   str    
 15  SYMBOL           0 non-null      float64
 16  TERMINATED       0 non-null      float64
 17  DECIMALS         6468 non

In [222]:
clean_df.columns = clean_df.columns.str.lower().str.replace(" ", "_")

In [223]:
clean_df["status"].value_counts(dropna=False)

status
NaN    5292
E       626
F       318
..      204
x        28
Name: count, dtype: int64

In [224]:
clean_df.query("status == 'F' & value.notnull()")["value"].count(), \
clean_df.query("status == '..' & value.notnull()")["value"].count(), \
clean_df.query("status == 'x' & value.notnull()")["value"].count()

(np.int64(0), np.int64(0), np.int64(0))

In [225]:
clean_df.query("status == 'E'")[["geo", "age_group", "gender", "characteristics", "value"]].groupby(["geo", "age_group", "gender", "characteristics"]).count()

value
geo          age_group                gender       characteristics                                       
Alberta      18 to 44 years           Both genders High 95% confidence interval, number of persons      1
                                                   Low 95% confidence interval, number of persons       1
                                                   Number of persons                                    1
                                      Men+         High 95% confidence interval, number of persons      2
                                                   Low 95% confidence interval, number of persons       2
...                                                                                                   ...
Saskatchewan 65 years and over        Women+       Low 95% confidence interval, number of persons       2
                                                   Number of persons                                    2
             Total, 18 years and over Men+         High 95% confidence interval, number of persons      2
                                                   Low 95% confidence interval, number of persons       2
                                                   Number of persons                                    2

[291 rows x 1 columns]

We are dropping the `status` column here because there is no obvious pattern or identifiable cause behind the data unreliability it indicates.

In [226]:
clean_df.count() == clean_df.query("scalar_factor == 'units'").count()

ref_date           True
geo                True
dguid              True
gender             True
age_group          True
indicators         True
characteristics    True
uom                True
uom_id             True
scalar_factor      True
scalar_id          True
vector             True
coordinate         True
value              True
status             True
symbol             True
terminated         True
decimals           True
dtype: bool

We are also dropping the `scalar_factor` column because all its values are `units`, so it does not provide any additional information.

In [227]:
clean_df = clean_df.drop(columns=["dguid", "uom", "uom_id", "scalar_id", "vector", "coordinate", "status", "symbol", "terminated", "decimals"])

In [228]:
clean_df = clean_df.pivot(index=["ref_date", "geo", "gender", "age_group", "indicators"], columns=["characteristics"], values="value").reset_index()
clean_df.columns.name = None

In [229]:
clean_df = clean_df.rename(columns={
    "Number of persons": "number",
    "Low 95% confidence interval, number of persons": "number_low_95",
    "High 95% confidence interval, number of persons": "number_high_95",
    "Percentage": "percentage",
    "Low 95% confidence interval, percentage": "percentage_low_95",
    "High 95% confidence interval, percentage": "percentage_high_95",
    "Statistically different from the Canada (excluding territories) rate": "different_from_canada_rate",
})

In [230]:
clean_df

,ref_date,geo,gender,age_group,indicators,number_high_95,percentage_high_95,number_low_95,percentage_low_95,number,percentage,different_from_canada_rate
0,2024,Alberta,Both genders,18 to 44 years,Life affected by wait time,408400.0,71.4,319500.0,60.9,363700.0,66.3,0.0
1,2024,Alberta,Both genders,18 to 44 years,Satisfaction with wait time - Dissatisfied or ...,242200.0,43.0,173900.0,32.5,206800.0,37.6,0.0
2,2024,Alberta,Both genders,18 to 44 years,Satisfaction with wait time - Very satisfied o...,258500.0,46.1,186600.0,35.1,222600.0,40.5,0.0
3,2024,Alberta,Both genders,18 to 44 years,"Total, wait time - medical specialist",594600.0,100.0,493100.0,100.0,546900.0,100.0,NaN
4,2024,Alberta,Both genders,18 to 44 years,"Wait time, 3 months to less than 6 months",112500.0,20.8,66400.0,12.8,89800.0,16.4,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
919,2024,Saskatchewan,Women+,"Total, 18 years and over",Satisfaction with wait time - Very satisfied o...,70200.0,48.1,55400.0,39.3,62900.0,43.6,-1.0
920,2024,Saskatchewan,Women+,"Total, 18 years and over","Total, wait time - medical specialist",154800.0,100.0,133000.0,100.0,144200.0,100.0,NaN
921,2024,Saskatchewan,Women+,"Total, 18 years and over","Wait time, 3 months to less than 6 months",34600.0,23.9,23400.0,16.8,29000.0,20.1,0.0
922,2024,Saskatchewan,Women+,"Total, 18 years and over","Wait time, 6 months or more",35000.0,24.1,24000.0,17.3,29500.0,20.5,0.0


In [231]:
clean_df.to_sql(
    "wait_times_clean",
    engine,
    if_exists="replace",
    index=False
)

-1